# Bayesian Blocks + background fit: iteration by iteration

`BayesianBlocksLightcurve.compute_bayesian_blocks()` (bc-tools) alternates two steps until the background intervals stop changing:

1. fit a polynomial background on the current background intervals (order 0 at iteration 0, 1 at iteration 1, 2 from then on);
2. run Bayesian Blocks with the exposure replaced by the background counts, find the peaks, and derive the signal window and the new background intervals (signal window ± `buffer_blocks` × width of the edge blocks).

bc-tools keeps only the last iteration. `bb_bkg_iterations` below is the same loop with every iteration recorded, so you can see where the fit goes bad. Section 3 checks that it reproduces the bc-tools result.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import peak_prominences, argrelmax

from gdt.core.background.binned import Polynomial
from gdt.core.binning.binned import rebin_by_edge_index, rebin_by_time
from gdt.core.data_primitives import TimeBins

from bctools.analysis import BayesianBlocksLightcurve
from bctools.analysis.bayesian_blocks import bayesian_blocks

from cosipy.nonimaging.bgo.ACSDataAnalyzer import ACSDataAnalyzer

## Parameters

In [ ]:
grb_file_name = 'bn210528586'
grb_fits_path = f'/data/grb_eliza/dc3_grbs/nobackground/1-10b/acs_fits/{grb_file_name}.fits'

# same arguments as BayesianBlocksLightcurve.compute_bayesian_blocks
p0 = 0.05
buffer_blocks = 5
signal_range = None   # optional first guess (tstart, tstop)
max_iter = 100

# plotting only
rebin_dt = None       # rebin the raw light curve in the plots (s), None = native bins
xlim = None           # (tmin, tmax) relative to the first bin, None = full range

## 1. Summed light curve

Same light curve that `ACSDataAnalyzer.analyze` gives to Bayesian Blocks: the sum of the six panels.

In [ ]:
light_curves = ACSDataAnalyzer().read_fits(grb_fits_path)

first = next(iter(light_curves.values()))
d = sum(np.asarray(lc.counts, dtype=float) for lc in light_curves.values())
lc_sum = TimeBins(d, first.lo_edges, first.hi_edges, first.exposure)

t0 = lc_sum.lo_edges[0]
print(f'{lc_sum.size} bins, width {np.median(lc_sum.exposure):.4g} s, '
      f'duration {lc_sum.hi_edges[-1] - t0:.1f} s, {d.sum():.0f} counts')

## 2. Iterations

In [ ]:
def bb_bkg_iterations(lc, p0=0.05, buffer_blocks=5, signal_range=None, max_iter=100):
    '''
    Same algorithm as BayesianBlocksLightcurve.compute_bayesian_blocks, returning one dict per iteration.
    The loop stops at the first iteration without peaks (marked with 'error') instead of raising.
    '''
    if signal_range is None:
        bkg_times = [(lc.range[0], lc.range[-1])]
    else:
        signal_range = np.sort(signal_range)
        bkg_times = [(lc.range[0], signal_range[0]), (signal_range[-1], lc.range[-1])]

    history, previous_bkg_times = [], []

    for i in range(max_iter):
        it = {'i': i, 'order': min(2, i), 'fit_times': bkg_times}

        # ---- background fit ----
        lc_bkg_times = TimeBins.merge([lc.slice(ti, tf) for ti, tf in bkg_times])
        bkg_model = Polynomial(counts=lc_bkg_times.counts[:, np.newaxis],
                               tstart=lc_bkg_times.lo_edges,
                               tstop=lc_bkg_times.hi_edges,
                               exposure=lc_bkg_times.exposure)
        bkg_model.fit(order=min(2, i))
        bkg_rate, _ = bkg_model.interpolate(lc.lo_edges, lc.hi_edges)
        bkg_counts = bkg_rate.flatten() * lc.exposure
        it['bkg_rate'] = bkg_rate.flatten()
        it['bkg_counts'] = bkg_counts

        # ---- Bayesian blocks ----
        lc_eff = TimeBins(counts=lc.counts, lo_edges=lc.lo_edges,
                          hi_edges=lc.hi_edges, exposure=bkg_counts)
        bb_index = bayesian_blocks(lc_eff, p0=p0)
        lc_bayes = lc.rebin(rebin_by_edge_index, bb_index)
        it['bb_index'] = bb_index
        it['lc_bayes'] = lc_bayes

        # ---- peaks and signal window ----
        peaks = argrelmax(lc_bayes.rates)[0]
        it['peaks'] = peaks
        if len(peaks) == 0:
            it['error'] = 'no peaks'
            history.append(it)
            break

        _, left_base, right_base = peak_prominences(lc_bayes.rates, peaks)
        new_start_signal = bb_index[np.min(left_base) + 1]
        new_stop_signal = bb_index[np.max(right_base)]
        it['signal'] = (lc.lo_edges[new_start_signal], lc.hi_edges[new_stop_signal])

        # ---- new background intervals ----
        block_widths = lc_bayes.widths
        left_buffer = min(buffer_blocks * block_widths[left_base[0] + 1],
                          (lc.lo_edges[new_start_signal] - lc.range[0]) / 2)
        right_buffer = min(buffer_blocks * block_widths[right_base[-1] - 1],
                           (lc.range[1] - lc.hi_edges[new_stop_signal]) / 2)
        new_start, new_stop = np.digitize([lc.lo_edges[new_start_signal] - left_buffer,
                                           lc.hi_edges[new_stop_signal - 1] + right_buffer],
                                          lc.lo_edges) - 1
        bkgex_tstart = max(lc.lo_edges[new_start], lc.lo_edges[bb_index[1]])
        bkgex_tstop = min(lc.hi_edges[new_stop], lc.hi_edges[bb_index[-2] - 1])
        bkg_times = [(lc.lo_edges[0], bkgex_tstart), (bkgex_tstop, lc.hi_edges[-1])]
        it['next_fit_times'] = bkg_times

        history.append(it)
        if bkg_times in previous_bkg_times and i >= 2:
            it['converged'] = True
            break
        previous_bkg_times.append(bkg_times)

    return history


def iterations_table(history, t0=0.):
    rows = []
    for it in history:
        nxt = it.get('next_fit_times')
        sig = it.get('signal', (np.nan, np.nan))
        rows.append({
            'iter': it['i'],
            'order': it['order'],
            'n_blocks': it['lc_bayes'].size,
            'n_peaks': len(it['peaks']),
            'signal_start': sig[0] - t0,
            'signal_stop': sig[1] - t0,
            'next_excl_start': nxt[0][1] - t0 if nxt else np.nan,
            'next_excl_stop': nxt[1][0] - t0 if nxt else np.nan,
            'bkg_rate_mean': it['bkg_rate'].mean(),
            'status': it.get('error', 'converged' if it.get('converged') else ''),
        })
    return pd.DataFrame(rows).set_index('iter')

In [ ]:
history = bb_bkg_iterations(lc_sum, p0=p0, buffer_blocks=buffer_blocks,
                            signal_range=signal_range, max_iter=max_iter)

if len(history) == max_iter and not history[-1].get('converged'):
    print(f'WARNING: reached max_iter = {max_iter} without converging')

# times relative to the first bin
iterations_table(history, t0)

### One panel per iteration

- grey: data; green shading: intervals used for the background fit **in this iteration**
- red dotted: fitted background; blue: Bayesian blocks; black triangles: peaks
- olive dashed: signal start/stop; purple: edges of the exclusion zone used in the **next** iteration

In [ ]:
def plot_bb_iterations(lc, history, rebin_dt=None, ncols=2, xlim=None, iterations=None):
    t0 = lc.lo_edges[0]
    lc_plot = lc.rebin(rebin_by_time, rebin_dt) if rebin_dt else lc
    its = history if iterations is None else [history[i] for i in iterations]

    nrows = int(np.ceil(len(its) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.5 * nrows),
                             sharex=True, squeeze=False)

    for ax, it in zip(axes.flat, its):
        ax.step(lc_plot.centroids - t0, lc_plot.rates, where='mid', color='.7', lw=0.8, label='data')

        for k, (ta, tb) in enumerate(it['fit_times']):
            ax.axvspan(ta - t0, tb - t0, color='tab:green', alpha=0.08,
                       label='bkg fit interval' if k == 0 else None)

        ax.plot(lc.centroids - t0, it['bkg_rate'], 'r:', lw=1.5, label=f"bkg (order {it['order']})")

        lb = it['lc_bayes']
        ax.plot(np.append(lb.lo_edges, lb.hi_edges[-1]) - t0, np.append(lb.rates, lb.rates[-1]),
                drawstyle='steps-post', color='tab:blue', label='BB')
        ax.plot(lb.centroids[it['peaks']] - t0, lb.rates[it['peaks']], 'kv', ms=5, label='peaks')

        if 'signal' in it:
            for t in it['signal']:
                ax.axvline(t - t0, ls='--', color='olive')
        if 'next_fit_times' in it:
            ax.axvline(it['next_fit_times'][0][1] - t0, color='purple', lw=0.8)
            ax.axvline(it['next_fit_times'][1][0] - t0, color='purple', lw=0.8)

        title = f"iter {it['i']}: {lb.size} blocks"
        if 'signal' in it:
            title += f", signal [{it['signal'][0] - t0:.2f}, {it['signal'][1] - t0:.2f}] s"
        if 'error' in it:
            title += f"  FAILED: {it['error']}"
        ax.set_title(title, fontsize=9)
        ax.set_ylabel('rate (cts/s)')
        if xlim:
            ax.set_xlim(xlim)

    for ax in axes.flat[len(its):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel(f't - {t0:.3f} (s)')
    axes.flat[0].legend(fontsize=7)
    fig.tight_layout()
    return fig


plot_bb_iterations(lc_sum, history, rebin_dt=rebin_dt, xlim=xlim)
plt.show()

### Convergence summary

Signal window, exclusion zone and number of blocks vs iteration. Jumps or a 2-3 iteration cycle show where the loop goes unstable.

In [ ]:
tab = iterations_table(history, t0)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))

axes[0].plot(tab.index, tab['signal_start'], 'o-', color='olive', label='signal start')
axes[0].plot(tab.index, tab['signal_stop'], 's-', color='olive', label='signal stop')
axes[0].plot(tab.index, tab['next_excl_start'], 'o:', color='purple', label='excl. start (next)')
axes[0].plot(tab.index, tab['next_excl_stop'], 's:', color='purple', label='excl. stop (next)')
axes[0].set_ylabel(f't - {t0:.3f} (s)')
axes[0].legend(fontsize=7)

axes[1].plot(tab.index, tab['n_blocks'], 'o-', label='blocks')
axes[1].plot(tab.index, tab['n_peaks'], 's-', label='peaks')
axes[1].legend(fontsize=7)

axes[2].plot(tab.index, tab['bkg_rate_mean'], 'o-', color='red')
axes[2].set_ylabel('mean bkg rate (cts/s)')

for ax in axes:
    ax.set_xlabel('iteration')
plt.tight_layout()
plt.show()

### Zoom on single iterations

Pick the iterations around the problem and zoom on the burst.

In [ ]:
zoom_iterations = list(range(max(0, len(history) - 4), len(history)))  # e.g. [0, 1, 2, 3]
zoom_xlim = None                                                        # e.g. (40, 80)

plot_bb_iterations(lc_sum, history, rebin_dt=rebin_dt, xlim=zoom_xlim, iterations=zoom_iterations)
plt.show()

## 3. Check against bc-tools

The last iteration must match `BayesianBlocksLightcurve.compute_bayesian_blocks` with the same arguments.

In [ ]:
last = history[-1]

try:
    bb = BayesianBlocksLightcurve(lc_sum)
    bb.compute_bayesian_blocks(p0=p0, buffer_blocks=buffer_blocks,
                               signal_range=signal_range, max_iter=max_iter)

    same_blocks = np.array_equal(bb._lc_bb_index, last['bb_index'])
    same_signal = np.allclose([bb.signal_range.tstart, bb.signal_range.tstop], last['signal'])
    same_bkg = np.allclose(bb.bkg_counts, last['bkg_counts'])
    print(f'same blocks: {same_blocks}, same signal range: {same_signal}, same background: {same_bkg}')

    fig, ax = plt.subplots(figsize=(12, 4))
    bb.plot(ax=ax, rebin_dt=rebin_dt)
    ax.set_title('bc-tools final result')
    if xlim:
        ax.set_xlim(xlim[0] + t0, xlim[1] + t0)
    plt.show()

except RuntimeError as exc:
    print(f'bc-tools failed: {exc}')
    print(f"debug loop: last iteration {last['i']}, status: {last.get('error', 'ok')}")